# 15 · Memory across conversations

Session state and long-term memory solve different problems. This demo indexes a completed session into InMemoryMemoryService and searches it from a new session.

**Runtime:** ADK 2.10.0 on `aadhi`, existing `dataengenv` kernel and ADC.

**Cost:** About 3 Lite calls; no managed memory charges. Synthetic ecommerce data only. Run cells from top to bottom; each notebook creates its own sessions.


In [ ]:
from pathlib import Path
import sys
# Works when Jupyter starts in GoogleADK or its parent folder.
root = Path.cwd() if (Path.cwd() / "course.py").exists() else Path.cwd() / "GoogleADK"
if not (root / "course.py").exists():
    raise RuntimeError("Open this notebook with GoogleADK as the working directory.")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
from course import (ROOT, CONFIG, MODEL, PROJECT, LOCATION, SKIP_SSL, DATA,
                    llm, gemini, make_lab, ask, final_text, token_usage,
                    get_order, get_product, get_customer, types)
print(f"Project={PROJECT} | location={LOCATION} | model={MODEL} | TLS bypass={SKIP_SSL}")


In [ ]:
from google.adk.memory import InMemoryMemoryService
from google.adk.tools import load_memory
memory = InMemoryMemoryService()
first = await make_lab(llm("preferences", "Acknowledge the customer preference briefly."), memory=memory)
await ask(first, "Remember: my delivery preference is weekend delivery.")
await memory.add_session_to_memory(first.session)

second = await make_lab(llm("recall", "Use load_memory to find the user's delivery preference. Search for delivery.", tools=[load_memory]), memory=memory)
await ask(second, "What is my delivery preference?")
matches = await memory.search_memory(app_name=second.runner.app_name, user_id=second.user_id, query="delivery")
print("Memory matches:", len(matches.memories))


This memory service is local and keyword-based; it does not become durable merely because it spans sessions. `preload_memory` is the alternative when memory should be retrieved before the model chooses a tool. FAISS document retrieval is covered separately.


## Try it

Repeat with a different user_id and verify that the first user's memory is not returned.


## Reference

[Official ADK documentation](https://adk.dev/sessions/memory/). Shared configuration: `config.json`. No environment activation or login cells are needed.
